# 梯度检查点与激活重算

一句话总结：反向传播需要用到激活结果，但是全量激活结果内存压力很大。于是乎采用检查点的策略，只存其中一部分，反向传播需要用到时重新算激活结果。

## 问题描述

训练 Transformer / LLM 时，显存大致被三块吃掉：

1. **参数**（以及 Adam 的动量等优化器状态）
2. **梯度**
3. **激活（activations）**：前向时每一层的中间张量，反传算 \(\partial L/\partial W\) 时要用

层数 \(L\)、序列长度、batch 一大，激活往往比参数更夸张。全量缓存 = 记得住所有层的输出；层越深，激活显存近似 \(O(L)\)。

## 基本概念

### 为什么反传需要激活

以一层 \(y = f(x; W)\) 为例，反传通常需要：

- 上游传来的 \(\partial L/\partial y\)
- 前向时的 \(x\)（有时还有层内中间量）

没有这些，就无法正确算 \(\partial L/\partial W\) 和 \(\partial L/\partial x\)。所以朴素 autograd 会在前向时把激活都挂在计算图上。

### 检查点在干什么

不要「每一层输出都长期留着」，而是：

1. 前向时按段推进；**只在分段边界保留 checkpoint 张量**（例如每个 Transformer block 的输入）
2. 段内中间激活 **立刻丢掉**（不占显存）
3. 反传到该段时：从最近的 checkpoint **再跑一遍该段前向**，临时重建激活 → 算梯度 → 再丢掉

这就是 **activation checkpointing / rematerialization（激活重算）**。

```
前向（省显存）:
  [ckpt0] → layer0..k-1（中间不存）→ [ckpt1] → layer k.. → [ckpt2] → ...

反向（到某一段）:
  取出 ckpt_i → 重算该段前向 → 用激活算梯度 → 释放临时激活
```

### 代价：用算力换显存

| | 全量存激活 | Checkpoint |
|---|---|---|
| 激活显存 | 高，约随层数线性涨 | 低；均匀分段时常到约 \(O(\sqrt{L})\) 量级 |
| 计算量 | 1× 前向 + 1× 反向 | 前向大约再多 **~1×**（每段多算一次） |
| 训练吞吐 | 更快（算得少） | 更慢，但能塞更大 batch / 更长序列 / 更深模型 |

经验口诀：**多付一次前向，换大幅激活显存。**

### 和易混概念的区别

| 名字 | 实际含义 |
|---|---|
| **Gradient checkpointing**（PyTorch `torch.utils.checkpoint`） | = 激活检查点 / 重算，**不是**数值梯度检验 |
| **Gradient checking** | 用有限差分验 \(\partial L/\partial W\) 对不对，调试用 |
| **ZeRO / CPU offload** | 切的是**参数、梯度、优化器状态**；和激活检查点正交，常一起用 |
| **推理** | 一般不需要 checkpoint；这是**训练反传**的技巧 |

### 实现注意（PyTorch）

```python
from torch.utils.checkpoint import checkpoint

# 把一段前向包起来；反传时自动重算
y = checkpoint(transformer_block, x, use_reentrant=False)
```

细节：

1. **分段粒度**：常见按 **一个 Transformer block** 做 checkpoint；太碎重算开销大，太粗省显存有限。
2. **Dropout / RNG**：段内有随机性时，重算必须复现同一随机性；`checkpoint` 会保存/恢复 RNG state。
3. **`use_reentrant`**：老默认 `True` 有若干边角问题；新代码推荐 **`use_reentrant=False`**。
4. **输入需要梯度**：被 checkpoint 的张量若完全不需要反传，行为要小心；训练主干输入通常 `requires_grad=True` 或依赖参数。
5. **和混合精度 / FSDP**：可叠加；注意不要重复保存同一份激活。

### 更完整的一句话

反传需要激活，全量缓存显存贵；checkpoint 只保留分段边界，反传时从边界重跑前向恢复段内激活，用 **额外计算换显存**。
